# 🚀 03. Model Training & Interaction Constraints
**Project**: XGBoost-Powered PE Malware Detection  
**Purpose**: Train and benchmark 5 diverse models (XGBoost, Random Forest, LightGBM, CatBoost, and IsolationForest+GB Hybrid) and evaluate the impact of Interaction Constraints.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

sys.path.insert(0, str(Path('../').resolve()))
from utils.preprocessing import MalwarePreprocessor, prepare_splits, compute_scale_pos_weight, ALL_FEATURES
from utils.evaluation import compute_all_metrics, benchmark_inference_time
from models.xgboost_model import XGBoostMalwareDetector
from models.random_forest_model import RandomForestMalwareDetector
from models.lightgbm_model import LightGBMMalwareDetector
from models.hybrid_model import HybridMalwareDetector

df = pd.read_csv('../data/synthetic_malware_data.csv')
train_df, val_df, test_df = prepare_splits(df)

preprocessor = MalwarePreprocessor(scaler_type='robust')
X_train = preprocessor.fit_transform(train_df)
X_val = preprocessor.transform(val_df)
X_test = preprocessor.transform(test_df)

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values
spw = compute_scale_pos_weight(train_df['label'])


## 1. Train Constrained XGBoost vs Unconstrained XGBoost (Ablation Study)


In [ ]:
xgb_constrained = XGBoostMalwareDetector(feature_names=ALL_FEATURES, with_constraints=True, scale_pos_weight=spw)
xgb_constrained.fit(X_train, y_train, X_val, y_val)
y_pred_c = xgb_constrained.predict(X_test)
y_prob_c = xgb_constrained.predict_proba(X_test)[:, 1]
metrics_c = compute_all_metrics(y_test, y_pred_c, y_prob_c, model_name="XGBoost (Constrained)")

xgb_unconstrained = XGBoostMalwareDetector(feature_names=ALL_FEATURES, with_constraints=False, scale_pos_weight=spw)
xgb_unconstrained.fit(X_train, y_train, X_val, y_val)
y_pred_u = xgb_unconstrained.predict(X_test)
y_prob_u = xgb_unconstrained.predict_proba(X_test)[:, 1]
metrics_u = compute_all_metrics(y_test, y_pred_u, y_prob_u, model_name="XGBoost (Unconstrained)")

pd.DataFrame([metrics_c, metrics_u]).set_index('model_name')[['roc_auc', 'f1_score', 'precision', 'recall', 'mcc']]


## 2. Train Benchmark Models


In [ ]:
rf = RandomForestMalwareDetector(feature_names=ALL_FEATURES)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
metrics_rf = compute_all_metrics(y_test, y_pred_rf, y_prob_rf, model_name="Random Forest")

lgb = LightGBMMalwareDetector(feature_names=ALL_FEATURES)
lgb.fit(X_train, y_train, X_val, y_val)
y_pred_lgb = lgb.predict(X_test)
y_prob_lgb = lgb.predict_proba(X_test)[:, 1]
metrics_lgb = compute_all_metrics(y_test, y_pred_lgb, y_prob_lgb, model_name="LightGBM")

hybrid = HybridMalwareDetector(feature_names=ALL_FEATURES)
hybrid.fit(X_train, y_train, X_val, y_val)
y_pred_hyb = hybrid.predict(X_test)
y_prob_hyb = hybrid.predict_proba(X_test)[:, 1]
metrics_hyb = compute_all_metrics(y_test, y_pred_hyb, y_prob_hyb, model_name="Hybrid (IF+GB)")

comparison_df = pd.DataFrame([metrics_c, metrics_rf, metrics_lgb, metrics_hyb]).set_index('model_name')
comparison_df[['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'pr_auc', 'mcc']]
